# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import load_pickle

from lightgbm import LGBMClassifier

from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
# cols_to_drop = ['lg_0', 'lg_1', 'lg_2', 'extra_0', 'extra_1', 'extra_2', 'rf_0', 'rf_1', 'rf_2', 'hist_0', 'hist_1', 'hist_2']

X_train = pd.read_parquet('../data/X_train_stacking_layer_two.parquet') #.drop(columns=cols_to_drop)
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_two.parquet') #.drop(columns=cols_to_drop)

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.999910,0.000083,0.000007,0.999866,0.000124,0.000010,0.999928,0.000070,0.000003,0.988210,0.004367,0.007423,0.999210,0.000790,0.000000
1,0.992961,0.000411,0.006628,0.991422,0.000567,0.008011,0.992562,0.000566,0.006872,0.986847,0.004779,0.008373,0.988522,0.001245,0.010233
2,0.000109,0.999858,0.000033,0.000162,0.999819,0.000019,0.000030,0.999962,0.000008,0.009897,0.988895,0.001208,0.002604,0.997396,0.000000
3,0.999805,0.000187,0.000009,0.999766,0.000223,0.000010,0.999837,0.000159,0.000004,0.988202,0.004372,0.007426,0.999098,0.000902,0.000000
4,0.998393,0.001560,0.000047,0.998544,0.001413,0.000043,0.998238,0.001711,0.000051,0.987966,0.004405,0.007629,0.998207,0.001793,0.000000


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.997841,0.002113,0.000045,0.997856,0.001945,0.000199,0.998030,0.001934,0.000036,0.987950,0.004507,0.007543,0.997783,0.002217,0.000000
1,0.996887,0.003080,0.000033,0.997599,0.002381,0.000020,0.996744,0.003240,0.000016,0.987886,0.004549,0.007565,0.997324,0.002676,0.000000
2,0.998244,0.001026,0.000731,0.998669,0.000563,0.000768,0.998349,0.000750,0.000901,0.987787,0.004504,0.007709,0.998949,0.001051,0.000000
3,0.000658,0.000086,0.999256,0.001490,0.000130,0.998380,0.000538,0.000138,0.999324,0.013634,0.004166,0.982199,0.000000,0.002127,0.997873
4,0.999840,0.000151,0.000009,0.999810,0.000178,0.000012,0.999880,0.000116,0.000004,0.988019,0.004456,0.007524,0.999069,0.000931,0.000000


# Machine Learning

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import optuna


def objective(trial, X, y):

    params = {
        "objective": "multiclass",
        "num_class": 3,
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 3000),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 100, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 10),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": 1,
        "verbosity": -1
    }

    w0 = trial.suggest_float("weight_class_0", 0.05, 10.0, log=True)
    w1 = trial.suggest_float("weight_class_1", 0.05, 10.0, log=True)
    w2 = trial.suggest_float("weight_class_2", 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = LGBMClassifier(**params)

        model.fit(
            X_train_fold,
            y_train_fold,
            eval_set=[(X_valid_fold, y_valid_fold)],
            eval_metric="multi_logloss",
        )

        proba = model.predict_proba(X_valid_fold)

        weighted_proba = proba * weights

        pred = np.argmax(weighted_proba, axis=1)

        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train.class_encoded
    ),
    n_trials=30,
    n_jobs=-1,
    show_progress_bar=True
)

print("Best score:", study.best_value)
print("Best params:")
print(study.best_params)

[I 2026-06-26 16:26:24,489] A new study created in memory with name: no-name-ff3394a4-6c7d-4465-9a8f-14bb071915bb
Best trial: 0. Best value: 0.847535:   3%|████                                                                                                                       | 1/30 [04:50<2:20:27, 290.61s/it]

[I 2026-06-26 16:31:15,089] Trial 0 finished with value: 0.8475349908982064 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.12577904944475982, 'n_estimators': 1205, 'num_leaves': 64, 'max_depth': 5, 'min_child_samples': 92, 'min_child_weight': 0.0071748666483829856, 'subsample': 0.5397306014086899, 'subsample_freq': 3, 'colsample_bytree': 0.6060338062341973, 'reg_alpha': 3.909483181132524, 'reg_lambda': 0.038473207246791485, 'min_split_gain': 4.306278140022404, 'weight_class_0': 5.065546283788345, 'weight_class_1': 1.1726432393473336, 'weight_class_2': 0.1247923374501062}. Best is trial 0 with value: 0.8475349908982064.


Best trial: 0. Best value: 0.847535:   7%|████████▏                                                                                                                  | 2/30 [06:00<1:14:55, 160.56s/it]

[I 2026-06-26 16:32:24,616] Trial 7 finished with value: 0.3333333333333333 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.0012924094117941778, 'n_estimators': 478, 'num_leaves': 129, 'max_depth': 6, 'min_child_samples': 123, 'min_child_weight': 71.32631989746245, 'subsample': 0.5855932347909096, 'subsample_freq': 5, 'colsample_bytree': 0.8619761978406313, 'reg_alpha': 6.0621644569280235e-05, 'reg_lambda': 2.3154065295828105e-06, 'min_split_gain': 4.009430187972862, 'weight_class_0': 0.9193283637885699, 'weight_class_1': 0.10835761490513311, 'weight_class_2': 0.23814253325499796}. Best is trial 0 with value: 0.8475349908982064.


Best trial: 0. Best value: 0.847535:  10%|████████████▎                                                                                                              | 3/30 [07:41<1:00:04, 133.51s/it]

...

[I 2026-06-26 19:09:29,985] Trial 9 finished with value: 0.9573597989670197 and parameters: {'boosting_type': 'dart', 'learning_rate': 0.009892567281834366, 'n_estimators': 985, 'num_leaves': 22, 'max_depth': -1, 'min_child_samples': 21, 'min_child_weight': 1.2407262462549489, 'subsample': 0.841055817987902, 'subsample_freq': 3, 'colsample_bytree': 0.6201013347805708, 'reg_alpha': 0.0009021408440429696, 'reg_lambda': 7.422038534138129e-05, 'min_split_gain': 0.5019314538253933, 'weight_class_0': 1.3550698215979757, 'weight_class_1': 0.5930492622568279, 'weight_class_2': 7.55558495051248}. Best is trial 8 with value: 0.9622489463672871.


Best trial: 8. Best value: 0.962249:  70%|███████████████████████████████████████████████████████████████████████████████████▎                                   | 21/30 [3:21:49<3:12:51, 1285.71s/it]

[I 2026-06-26 19:48:13,587] Trial 15 pruned. 


Best trial: 8. Best value: 0.962249:  73%|███████████████████████████████████████████████████████████████████████████████████████▎                               | 22/30 [4:11:41<3:59:42, 1797.79s/it]

[I 2026-06-26 20:38:05,548] Trial 12 pruned. 


In [7]:
# study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=500, n_jobs=-1, show_progress_bar=True)

# print("\nBest trial score:")
# print(study.best_trial.value)

# print("\nBest params:")
# print(study.best_trial.params)

In [8]:
sgd_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

sgd = LGBMClassifier(
    **params,
    objective="multiclass",
    num_class=3,
    random_state=42,
    n_jobs=1,
    verbosity=-1
).fit(X_train, y_train.class_encoded)

loss = sgd_params['loss']

if loss in ["log_loss", "modified_huber"]:
    test_proba = sgd.predict_proba(X_test)

else:
    test_proba = sgd.decision_function(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [9]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [10]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_sgd.csv', index=False)

In [11]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [12]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'lg_0', 'lg_1', 'lg_2', 'sgd_0', 'sgd_1', 'sgd_2'],
      dtype='str')

In [13]:
len(study.trials)

60